In [1]:
import os 
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/train',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/val',
    label_mode='int',
    image_size=(224, 224),
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    '../artifacts/data/processed/test',
    label_mode='int',
    image_size=(224, 224),
    shuffle=False
)

Found 10363 files belonging to 2 classes.
Found 3157 files belonging to 2 classes.
Found 3138 files belonging to 2 classes.


In [3]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
])


In [4]:
base_model = tf.keras.applications.MobileNetV2(
            weights='imagenet',
            input_shape=(224, 224, 3),
            include_top=False
        )

In [ ]:
base_model.trainable = False

num_classes = 2


model = models.Sequential([
    layers.InputLayer(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

d:\SAMITH\Github\Image-Based-Food-Freshness-Prediction-System\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [6]:
model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=["accuracy"]
        )

In [7]:
early_stopping = EarlyStopping(
            monitor='val_loss',    
            patience=5,
            restore_best_weights=True
        )

In [8]:
history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=50,
            callbacks=[early_stopping]  
        )

Epoch 1/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 425s 1s/step - accuracy: 0.9134 - loss: 0.2115 - val_accuracy: 0.9566 - val_loss: 0.1159
Epoch 2/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 483s 1s/step - accuracy: 0.9559 - loss: 0.1166 - val_accuracy: 0.9636 - val_loss: 0.1040
Epoch 3/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 455s 1s/step - accuracy: 0.9622 - loss: 0.0964 - val_accuracy: 0.9743 - val_loss: 0.0757
Epoch 4/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 452s 1s/step - accuracy: 0.9718 - loss: 0.0752 - val_accuracy: 0.9690 - val_loss: 0.0858
Epoch 5/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 442s 1s/step - accuracy: 0.9698 - loss: 0.0759 - val_accuracy: 0.9753 - val_loss: 0.0752
Epoch 6/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 421s 1s/step - accuracy: 0.9793 - loss: 0.0596 - val_accuracy: 0.9807 - val_loss: 0.0560
Epoch 7/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 466s 1s/step - accuracy: 0.9783 - loss: 0.0558 - val_accuracy: 0.9823 - val_loss: 0.0512
Epoch 8/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 436s 1s/step - accuracy: 0.9812 - loss: 0.0497 - val_accu

In [9]:

os.makedirs('artifacts/models', exist_ok=True) 
model.save('artifacts/models/with_augmentation.keras')


In [5]:
from keras.callbacks import ReduceLROnPlateau

In [6]:
base_model.trainable = False

for layer in base_model.layers[:-10]:  
    layer.trainable = False

num_classes = 2


model = models.Sequential([
    layers.InputLayer(shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

In [7]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [8]:
early_stopping = EarlyStopping(
            monitor='val_loss',    
            patience=5,
            restore_best_weights=True
        )

In [9]:
lr_schedule = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stopping, lr_schedule]
)

Epoch 1/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 512s 2s/step - accuracy: 0.7810 - loss: 0.4516 - val_accuracy: 0.8929 - val_loss: 0.2927 - learning_rate: 5.0000e-05
Epoch 2/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 417s 1s/step - accuracy: 0.9033 - loss: 0.2561 - val_accuracy: 0.9202 - val_loss: 0.2153 - learning_rate: 5.0000e-05
Epoch 3/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 396s 1s/step - accuracy: 0.9258 - loss: 0.2046 - val_accuracy: 0.9344 - val_loss: 0.1813 - learning_rate: 5.0000e-05
Epoch 4/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 377s 1s/step - accuracy: 0.9347 - loss: 0.1767 - val_accuracy: 0.9424 - val_loss: 0.1604 - learning_rate: 5.0000e-05
Epoch 5/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 375s 1s/step - accuracy: 0.9417 - loss: 0.1583 - val_accuracy: 0.9471 - val_loss: 0.1471 - learning_rate: 5.0000e-05
Epoch 6/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 373s 1s/step - accuracy: 0.9463 - loss: 0.1493 - val_accuracy: 0.9522 - val_loss: 0.1385 - learning_rate: 5.0000e-05
Epoch 7/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 373s 1s/step - acc

In [10]:
os.makedirs('artifacts/models', exist_ok=True) 
model.save('artifacts/models/with_fine_tunning.keras')

In [11]:
from tensorflow.keras import regularizers

In [12]:
num_classes = 2

model = models.Sequential([
    layers.InputLayer(input_shape=(224, 224, 3)),
    data_augmentation,
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),  
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),  
    layers.Dropout(0.3),  
    layers.Dense(num_classes, activation='softmax')
])

d:\SAMITH\Github\Image-Based-Food-Freshness-Prediction-System\venv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

In [18]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 406s 1s/step - accuracy: 0.7058 - loss: 0.6033 - val_accuracy: 0.8841 - val_loss: 0.3316 - learning_rate: 1.0000e-04
Epoch 2/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 386s 1s/step - accuracy: 0.8420 - loss: 0.3658 - val_accuracy: 0.9145 - val_loss: 0.2503 - learning_rate: 1.0000e-04
Epoch 3/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 381s 1s/step - accuracy: 0.8730 - loss: 0.3109 - val_accuracy: 0.9256 - val_loss: 0.2138 - learning_rate: 1.0000e-04
Epoch 4/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 379s 1s/step - accuracy: 0.8867 - loss: 0.2802 - val_accuracy: 0.9344 - val_loss: 0.1914 - learning_rate: 1.0000e-04
Epoch 5/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 369s 1s/step - accuracy: 0.8985 - loss: 0.2604 - val_accuracy: 0.9370 - val_loss: 0.1798 - learning_rate: 1.0000e-04
Epoch 6/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 367s 1s/step - accuracy: 0.9072 - loss: 0.2410 - val_accuracy: 0.9439 - val_loss: 0.1669 - learning_rate: 1.0000e-04
Epoch 7/50
324/324 ━━━━━━━━━━━━━━━━━━━━ 368s 1s/step - acc

In [19]:
os.makedirs('./artifacts/models', exist_ok=True)
model.save('./artifacts/models/with_regularization.keras')

In [20]:
model = tf.keras.models.load_model(
    r'D:\SAMITH\Github\Image-Based-Food-Freshness-Prediction-System\artifacts\models\mobilenetv2_baseline.keras'
)

In [21]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

99/99 ━━━━━━━━━━━━━━━━━━━━ 90s 877ms/step - accuracy: 0.9847 - loss: 0.0576
Test accuracy: 0.9847 | Test loss: 0.0576
